## Handwritten Digit Recognition - MNIST Dataset - CNN(Convolutional Neural Network) - Python/Keras

#### Key Design Considerations

This is a Multi-Class Classification problem (10 classes)
- Language: Python
- Deep Learning Package: Keras
- Dataset: MNIST dataset available with Keras
- Model: CNN
- We save the model in tfilte format, both in its normal and quantized form
- These two versions of the model can be used directly in Android with the support of the tflite library.

#### Load Dataset

Load train and test datasets

In [2]:
from keras.datasets import mnist

In [3]:
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [4]:
train_images.shape

(60000, 28, 28)

In [5]:
train_labels.shape

(60000,)

In [6]:
test_images.shape

(10000, 28, 28)

In [7]:
test_labels.shape

(10000,)

<br>**Training Set**
- 60000 images
- Each image is of the shape 28 x 28 (rows x columns)
- 60000 labels defining the digit that corresponds to the respective image

<br>**Test Set**
- 10000 images
- Each image is of the shape 28 x 28 (rows x columns)
- 10000 labels defining the digit that corresponds to the respective image

#### Layer Details:
- 2 dimensional Convolution Layer
- Number of filters/kernels = 32
- Filter/Kernel Size = 3x3
- Activation Function = relu (for non-linearity detection)
- Input Shape = 28x28 matrix with 1 channel (as image is gray scale, we have only 1 channel)

In [8]:
from keras import models

In [9]:
from keras import layers

In [10]:
model_cnn = models.Sequential()

In [11]:
model_cnn.add(layers.Conv2D(32, (3,3), activation='relu', input_shape=(28,28,1)))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


#### Layer Details:

- Downsample the output from previous layer
- We will take the max value for a every 2x2 window ... moved over the input

In [12]:
model_cnn.add(layers.MaxPooling2D(2,2))

### Layer Details:

- 2 dimensional Convolution Layer
- Number of filters/kernels = 64
- Filter/Kernel Size = 3x3
- Activation Function = relu (for non-linearity detection)

In [13]:
model_cnn.add(layers.Conv2D(64, (3,3), activation = 'relu'))

### Layer Details:

- Downsample the output from previous layer
- We will take the max value for a every 2x2 window ... moved over the input

In [14]:
model_cnn.add(layers.MaxPooling2D(2,2))

### Layer Details:

- 2 dimensional Convolution Layer
- Number of filters/kernels = 64
- Filter/Kernel Size = 3x3
- Activation Function = relu (for non-linearity detection)

In [15]:
model_cnn.add(layers.Conv2D(64, (3,3), activation='relu'))

Data at this stage is in matrix form. We will convert it to vector form to feed to a fully connected network (FCN).

In [16]:
model_cnn.add(layers.Flatten())

We will design for 64 outputs with activation function as relu (to learn non-linearity).

In [17]:
model_cnn.add(layers.Dense(64, activation = 'relu'))

This is the final layer. Hence, the outputs will be 10 corresponding to the 10 digits (0 to 9). Activation Function chosen here is softmax to have a probabilistic output.

In [18]:
model_cnn.add(layers.Dense(10, activation = 'softmax'))

In [19]:
model_cnn.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 3, 3, 64)       │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 576)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 93,322 (364.54 KB)

 Trainable params: 93,322 (364.54 KB)

 Non-trainable params: 0 (0.00 B)

Data Preprocessing - Train and Test Images

In [20]:
train_images.shape

(60000, 28, 28)

CNN needs another dimension for the channel. Here as the image is gray scale it will be 1 channel. If we had color images, the channel value would have been 3 for the three channels - Red, Green and Blue.

In [21]:
train_images_cnn = train_images.reshape(60000, 28, 28, 1)

We need to change the element values from integer to decimal to have continuos values during prediction through the various layers. We will limit the values to the interval [0,1] so that the model treats each sample with equal weightage as the range of values for all samples will be fixed. We will do this by dividing the decimal values by 255 (gray scale values are from 0 to 255 ... 0 representing black to white).

In [24]:
train_images_cnn = train_images_cnn.astype('float32') / 255

In [25]:
test_images_cnn = test_images.reshape(10000, 28, 28, 1)

In [26]:
test_images_cnn = test_images_cnn.astype('float32') / 255

#### **Data Preprocessing - Train and Test Labels**
We will convert the labels to 10bit values. Only 1 of the bits of the 10bit value will be 1 corresponding to the location for the respective digit and rest all bits will be 0. This is required to match to the model's output layer expectation so that we can effectively train and test.

In [27]:
from keras.utils import to_categorical

In [28]:
train_labels_cnn = to_categorical(train_labels)

In [29]:
test_labels_cnn = to_categorical(test_labels)

#### **Define the optimizer function, loss function and metrics to be used for the model.**
Going ahead with the well known functions at this point in time
Selected accuracy as the metrics to understand validation / test accuracy of the model

In [32]:
model_cnn.compile(optimizer='rmsprop', loss='categorical_crossentropy', metrics=['accuracy'])

**Train the Model**
**We will now train the model using train images and train labels.**
- We will use a batch size = 60.
- 1 epoch = 60000 / 60 = 1000 batches
- 1 epoch = 1 complete run of all train samples for training the model
- We will go for a total of 5 epochs = 5 complete run of the all train samples

In [33]:
model_cnn.fit(train_images_cnn, train_labels_cnn, epochs = 5, batch_size = 60)

Epoch 1/5
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 58s 55ms/step - accuracy: 0.9484 - loss: 0.1654
Epoch 2/5
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 56s 56ms/step - accuracy: 0.9863 - loss: 0.0442
Epoch 3/5
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 55s 55ms/step - accuracy: 0.9904 - loss: 0.0308
Epoch 4/5
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 82s 55ms/step - accuracy: 0.9927 - loss: 0.0234
Epoch 5/5
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 55s 55ms/step - accuracy: 0.9946 - loss: 0.0178


**Test the Model**
### We will now test model's performance with the test data.

- We predict the class for each of the 10000 test using the model.
- We will check the test accuracy.

In [34]:
test_loss_cnn, test_acc_cnn = model_cnn.evaluate(test_images_cnn, test_labels_cnn)

313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - accuracy: 0.9902 - loss: 0.0332


In [35]:
print('test accuracy:', (test_acc_cnn*100))

test accuracy: 99.01999831199646


Converting the Keras model to a tflite model

In [37]:
import tensorflow as tf

In [38]:
# Convert Model to TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(model_cnn)
tfmodel = converter.convert()
open("digit_recognition_cnn.tflite","wb").write(tfmodel)

Saved artifact at '/tmp/tmpy4ld8t7k'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 28, 28, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  132308892545296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132308892546448: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132308892545872: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132308892547216: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132308892548176: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132308892548560: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132308820165840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132308820166608: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132308820165648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132308820166992: TensorSpec(shape=(), dtype=tf.resource, name=None)


377820

In [39]:
from google.colab import files

files.download("digit_recognition_cnn.tflite")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [40]:
# Convert Model to TFLite and Apply Quantization
converter1 = tf.lite.TFLiteConverter.from_keras_model(model_cnn)
converter1.optimizations = [tf.lite.Optimize.DEFAULT]
tfmodel1 = converter1.convert()
open("digit_recognition_cnn_quant.tflite","wb").write(tfmodel1)

Saved artifact at '/tmp/tmpgjim2416'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 28, 28, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  132308892545296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132308892546448: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132308892545872: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132308892547216: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132308892548176: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132308892548560: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132308820165840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132308820166608: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132308820165648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132308820166992: TensorSpec(shape=(), dtype=tf.resource, name=None)


103616

In [41]:


from google.colab import files

files.download("digit_recognition_cnn_quant.tflite")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>